# 08 — Atmospheric Scientific Analysis

Derived meteorological diagnostics computed from raw GFS fields:

| Analysis | Derived from | Physical meaning |
|---|---|---|
| Relative vorticity | u10, v10 | Rotation of the wind field — cyclones/anticyclones |
| Horizontal divergence | u10, v10 | Air convergence (inflow) vs divergence (outflow) |
| Moisture flux convergence | u10, v10, sh2 | Where moist air converges → precipitation trigger |
| Wind speed & direction | u10, v10 | Magnitude and bearing |
| Convective instability | CAPE, CIN | Thunderstorm potential |
| Precipitable water time series | pwat | Total column water vapour over a region |

---
**Stack:** `noawclg`, `xarray`, `numpy`, `scipy`, `matplotlib`, `cartopy`, `cmocean`

In [ ]:
import noawclg
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.ticker as mticker
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
import cmocean
import copy

In [ ]:
BG = "#0d1117"; LAND = "#13171d"; OCEAN_C = "#0d1520"
COAST = "#2d6db5"; BORDER = "#2a2f36"; STATE = "#1e2530"
TEXT = "#e6edf3"; SUBTEXT = "#8b949e"
BOXBG = "#161b22"; BOXEDGE = "#30363d"; GRID = "#14181c"

plt.rcParams.update({
    "figure.facecolor":  BG,
    "axes.facecolor":    BG,
    "savefig.facecolor": BG,
    "text.color":        TEXT,
    "axes.labelcolor":   SUBTEXT,
    "xtick.color":       SUBTEXT,
    "ytick.color":       SUBTEXT,
    "axes.edgecolor":    BOXEDGE,
    "axes.grid":         True,
    "grid.color":        GRID,
    "grid.linewidth":    0.5,
    "font.family":       "monospace",
    "legend.facecolor":  BOXBG,
    "legend.edgecolor":  BOXEDGE,
    "legend.labelcolor": TEXT,
})

RUN_DATE = "20260418"
CYCLE    = "00"
F_HOURS  = range(0, 121, 3)   # 41 frames

## 0  Load all fields at once

In [ ]:
def load(var, hours=None):
    return noawclg.load(var=var, run_date=RUN_DATE, cycle=CYCLE,
                         forecast_hours=hours or F_HOURS)

ds_u  = load("u10")
ds_v  = load("v10")
ds_sh = load("sh2")
ds_pw = load("pwat")
ds_cp = load("cape")
ds_ci = load("cin")
ds_t  = load("t2m")

lat = ds_u["lat"].values
lon = ds_u["lon"].values

# Convert lon 0–360 → −180/180
lon_180 = np.where(lon > 180, lon - 360, lon)
order   = np.argsort(lon_180)
lon_180 = lon_180[order]

def extract(ds, var, tidx=0, conv=None):
    """Pull one time step, reorder longitude, apply optional converter."""
    f = ds[var].isel(time=tidx).values[:, order]
    return f if conv is None else conv(f)

TIDX = 0   # change to any forecast step 0–40
date_label = str(ds_u["u10"].time.values[TIDX])[:16].replace("T", " ")

U   = extract(ds_u,  "u10")
V   = extract(ds_v,  "v10")
SH  = extract(ds_sh, "sh2")
PW  = extract(ds_pw, "pwat")
CP  = extract(ds_cp, "cape")
CI  = extract(ds_ci, "cin")
T2M = extract(ds_t,  "t2m", conv=lambda f: f - 273.15)

print(f"All fields loaded for {date_label}")

---
## 1  Wind Speed and Direction

$$\text{speed} = \sqrt{u^2 + v^2}, \qquad \text{direction} = \arctan2(-u, -v) \cdot \frac{180}{\pi}$$

Direction follows meteorological convention: 0° = wind from the North, 90° = from the East.

In [ ]:
wspd = np.sqrt(U**2 + V**2)
wdir = np.degrees(np.arctan2(-U, -V)) % 360   # meteorological convention

levels_ws = np.arange(0, 26, 1)
cmap_ws   = copy.copy(cmocean.cm.speed)
cmap_ws.set_under(alpha=0); cmap_ws.set_bad(alpha=0)
norm_ws   = mcolors.BoundaryNorm(levels_ws, ncolors=cmap_ws.N, clip=False)

# Region: South America
W, E, S, N = -85, -30, -60, 15

proj = ccrs.PlateCarree()
fig1, ax1 = plt.subplots(figsize=(12, 9),
                          subplot_kw={"projection": proj}, facecolor=BG)
ax1.set_facecolor(BG)
ax1.set_extent([W, E, S, N], crs=ccrs.PlateCarree())

ax1.add_feature(cfeature.OCEAN.with_scale("50m"),  facecolor=OCEAN_C, zorder=0)
ax1.add_feature(cfeature.LAND.with_scale("50m"),   facecolor=LAND,    zorder=0)
ax1.add_feature(cfeature.COASTLINE.with_scale("50m"),
                edgecolor=COAST, linewidth=0.6, zorder=3)
ax1.add_feature(cfeature.BORDERS.with_scale("50m"),
                edgecolor=BORDER, linewidth=0.4, zorder=3)
ax1.add_feature(
    cfeature.NaturalEarthFeature("cultural",
        "admin_1_states_provinces_lines", "10m",
        facecolor="none", edgecolor=STATE, linewidth=0.3),
    zorder=3,
)

cf1 = ax1.pcolormesh(lon_180, lat, wspd,
                     cmap=cmap_ws, norm=norm_ws,
                     transform=ccrs.PlateCarree(), zorder=1)

# Wind barbs — subsample every 8th point
step = 8
ax1.barbs(
    lon_180[::step], lat[::step],
    U[::step, ::step], V[::step, ::step],
    transform=ccrs.PlateCarree(),
    length=4, linewidth=0.5,
    color="#e6edf3", alpha=0.6,
    zorder=4,
)

gl1 = ax1.gridlines(draw_labels=True, linewidth=0.3, color=GRID, linestyle="--")
gl1.top_labels = False; gl1.right_labels = False
gl1.xlabel_style = {"color": SUBTEXT, "fontsize": 7}
gl1.ylabel_style = {"color": SUBTEXT, "fontsize": 7}

cb1 = fig1.colorbar(cf1, ax=ax1, orientation="horizontal",
                    pad=0.05, fraction=0.03, shrink=0.7)
cb1.set_label("10m Wind Speed (m s⁻¹)", fontsize=8, color=SUBTEXT)
cb1.ax.tick_params(labelsize=7, colors=SUBTEXT)
cb1.outline.set_edgecolor(BOXEDGE)

ax1.set_title("10m Wind Speed + Barbs — South America",
              fontsize=10, fontweight="bold", color=TEXT, loc="left", pad=6)
ax1.set_title(f"GFS {CYCLE}z  {date_label}",
              fontsize=7, color=SUBTEXT, loc="right", pad=6)

fig1.tight_layout()
fig1.savefig("wind_speed_barbs.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 2  Relative Vorticity

$$\zeta = \frac{\partial v}{\partial x} - \frac{\partial u}{\partial y}$$

Positive ζ (cyclonic in the NH) indicates counter-clockwise rotation; negative ζ indicates anticyclonic rotation.  
Units: s⁻¹ (typical values ×10⁻⁵ s⁻¹).

We use `numpy.gradient` with **physical distances** computed from the lat/lon grid.

In [ ]:
# Earth radius (m)
R_EARTH = 6_371_000.0

# Convert lat/lon from degrees to radians
lat_rad = np.radians(lat)
lon_rad = np.radians(lon_180)

# Grid spacings in metres
dlat = np.gradient(lat_rad) * R_EARTH                           # (n_lat,)
dlon = np.gradient(lon_rad) * R_EARTH * np.cos(lat_rad)[:, None]  # (n_lat, n_lon) — varies with lat
# numpy.gradient returns the spacing array; we need a scalar estimate per row/col
dlat_mean = np.mean(np.abs(np.gradient(lat_rad))) * R_EARTH

# dV/dx (longitude direction)
dvdx = np.gradient(V, axis=1) / np.mean(np.abs(np.gradient(lon_rad) * R_EARTH * np.cos(lat_rad[:, None])))

# dU/dy (latitude direction)
dudy = np.gradient(U, axis=0) / dlat_mean

vort = dvdx - dudy   # relative vorticity (s⁻¹)
vort_scaled = vort * 1e5   # scale to ×10⁻⁵ s⁻¹ for readability

print(f"Vorticity range: {vort_scaled.min():.2f} … {vort_scaled.max():.2f}  ×10⁻⁵ s⁻¹")

In [ ]:
levels_vort = np.arange(-8, 8.5, 0.5)
cmap_v      = copy.copy(cmocean.cm.curl)
cmap_v.set_bad(alpha=0)
norm_v      = mcolors.BoundaryNorm(levels_vort, ncolors=cmap_v.N, clip=False)

proj2 = ccrs.Orthographic(central_longitude=0, central_latitude=30)
fig2, ax2 = plt.subplots(figsize=(10, 10),
                          subplot_kw={"projection": proj2}, facecolor=BG)
ax2.set_facecolor(BG)
ax2.set_global()

ax2.add_feature(cfeature.LAND.with_scale("110m"),     facecolor=LAND,  zorder=2)
ax2.add_feature(cfeature.COASTLINE.with_scale("110m"),
                edgecolor=COAST, linewidth=0.5, zorder=3)
ax2.add_feature(cfeature.BORDERS.with_scale("110m"),
                edgecolor=BORDER, linewidth=0.3, zorder=3)

cf2 = ax2.pcolormesh(lon_180, lat, vort_scaled,
                     cmap=cmap_v, norm=norm_v,
                     transform=ccrs.PlateCarree(), zorder=1)

cb2 = fig2.colorbar(cf2, ax=ax2, orientation="horizontal",
                    pad=0.03, fraction=0.03, shrink=0.85)
cb2.set_label("Relative Vorticity (×10⁻⁵ s⁻¹)", fontsize=8, color=SUBTEXT)
cb2.ax.tick_params(labelsize=7, colors=SUBTEXT)
cb2.outline.set_edgecolor(BOXEDGE)

ax2.set_title("10m Relative Vorticity  (+= cyclonic NH, −= anticyclonic)",
              fontsize=9, fontweight="bold", color=TEXT, loc="left", pad=6)
ax2.set_title(f"GFS {CYCLE}z  {date_label}",
              fontsize=7, color=SUBTEXT, loc="right", pad=6)

fig2.tight_layout()
fig2.savefig("vorticity_globe.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 3  Horizontal Divergence

$$D = \frac{\partial u}{\partial x} + \frac{\partial v}{\partial y}$$

Positive D = divergence (air spreading out, e.g. anticyclone surface).  
Negative D = convergence (air piling up → vertical motion → convection).

In [ ]:
dudx = np.gradient(U, axis=1) / np.mean(np.abs(np.gradient(lon_rad) * R_EARTH * np.cos(lat_rad[:, None])))
dvdy = np.gradient(V, axis=0) / dlat_mean

div         = dudx + dvdy
div_scaled  = div * 1e5   # ×10⁻⁵ s⁻¹

levels_div = np.arange(-4, 4.5, 0.5)
cmap_d     = copy.copy(cmocean.cm.balance)
cmap_d.set_bad(alpha=0)
norm_d     = mcolors.BoundaryNorm(levels_div, ncolors=cmap_d.N, clip=False)

proj3 = ccrs.PlateCarree()
fig3, ax3 = plt.subplots(figsize=(16, 7),
                          subplot_kw={"projection": proj3}, facecolor=BG)
ax3.set_facecolor(BG)
ax3.set_global()

ax3.add_feature(cfeature.LAND.with_scale("110m"),     facecolor=LAND,  zorder=2)
ax3.add_feature(cfeature.COASTLINE.with_scale("110m"),
                edgecolor=COAST, linewidth=0.4, zorder=3)

cf3 = ax3.pcolormesh(lon_180, lat, div_scaled,
                     cmap=cmap_d, norm=norm_d,
                     transform=ccrs.PlateCarree(), zorder=1)

# Zero-line contour
ax3.contour(lon_180, lat, div_scaled,
            levels=[0], colors=["#e6edf3"],
            linewidths=0.6, alpha=0.5,
            transform=ccrs.PlateCarree(), zorder=4)

cb3 = fig3.colorbar(cf3, ax=ax3, orientation="horizontal",
                    pad=0.04, fraction=0.025, shrink=0.7)
cb3.set_label("Divergence (×10⁻⁵ s⁻¹)  —  blue = convergence, red = divergence",
              fontsize=8, color=SUBTEXT)
cb3.ax.tick_params(labelsize=7, colors=SUBTEXT)
cb3.outline.set_edgecolor(BOXEDGE)

ax3.set_title("10m Horizontal Divergence",
              fontsize=10, fontweight="bold", color=TEXT, loc="left", pad=6)
ax3.set_title(f"GFS {CYCLE}z  {date_label}",
              fontsize=7, color=SUBTEXT, loc="right", pad=6)

fig3.tight_layout()
fig3.savefig("divergence_global.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 4  Moisture Flux Convergence (MFC)

$$\text{MFC} = -\nabla \cdot (q \vec{V}) = -\left(\frac{\partial(qu)}{\partial x} + \frac{\partial(qv)}{\partial y}\right)$$

Positive MFC (convergence of moist air) → moisture accumulates → convection trigger.  
This is one of the best single-field predictors of precipitation location.

In [ ]:
qu = SH * U
qv = SH * V

dqudx = np.gradient(qu, axis=1) / np.mean(np.abs(np.gradient(lon_rad) * R_EARTH * np.cos(lat_rad[:, None])))
dqvdy = np.gradient(qv, axis=0) / dlat_mean

mfc = -(dqudx + dqvdy)   # kg kg⁻¹ m⁻¹ s⁻¹ — positive = moisture convergence
mfc_scaled = mfc * 1e7   # scale for readability

levels_mfc = np.arange(-5, 5.5, 0.5)
cmap_m     = copy.copy(cmocean.cm.balance)
cmap_m.set_bad(alpha=0)
norm_m     = mcolors.BoundaryNorm(levels_mfc, ncolors=cmap_m.N, clip=False)

W2, E2, S2, N2 = -85, -30, -60, 15   # South America

proj4 = ccrs.PlateCarree()
fig4, ax4 = plt.subplots(figsize=(12, 9),
                          subplot_kw={"projection": proj4}, facecolor=BG)
ax4.set_facecolor(BG)
ax4.set_extent([W2, E2, S2, N2], crs=ccrs.PlateCarree())

ax4.add_feature(cfeature.OCEAN.with_scale("50m"),  facecolor=OCEAN_C, zorder=0)
ax4.add_feature(cfeature.LAND.with_scale("50m"),   facecolor=LAND,    zorder=0)
ax4.add_feature(cfeature.COASTLINE.with_scale("50m"),
                edgecolor=COAST, linewidth=0.6, zorder=3)
ax4.add_feature(cfeature.BORDERS.with_scale("50m"),
                edgecolor=BORDER, linewidth=0.4, zorder=3)
ax4.add_feature(
    cfeature.NaturalEarthFeature("cultural",
        "admin_1_states_provinces_lines", "10m",
        facecolor="none", edgecolor=STATE, linewidth=0.3),
    zorder=3,
)

cf4 = ax4.pcolormesh(lon_180, lat, mfc_scaled,
                     cmap=cmap_m, norm=norm_m,
                     transform=ccrs.PlateCarree(), zorder=1)

# Moisture flux vectors
step = 10
ax4.quiver(
    lon_180[::step], lat[::step],
    qu[::step, ::step] * 100, qv[::step, ::step] * 100,
    transform=ccrs.PlateCarree(),
    scale=3, width=0.003,
    color="#e6edf3", alpha=0.5, zorder=4,
)

gl4 = ax4.gridlines(draw_labels=True, linewidth=0.3, color=GRID, linestyle="--")
gl4.top_labels = False; gl4.right_labels = False
gl4.xlabel_style = {"color": SUBTEXT, "fontsize": 7}
gl4.ylabel_style = {"color": SUBTEXT, "fontsize": 7}

cb4 = fig4.colorbar(cf4, ax=ax4, orientation="horizontal",
                    pad=0.05, fraction=0.03, shrink=0.7)
cb4.set_label("Moisture Flux Convergence (×10⁻⁷ kg kg⁻¹ m⁻¹ s⁻¹)  +  arrows = qV",
              fontsize=8, color=SUBTEXT)
cb4.ax.tick_params(labelsize=7, colors=SUBTEXT)
cb4.outline.set_edgecolor(BOXEDGE)

ax4.set_title("Moisture Flux Convergence — South America",
              fontsize=10, fontweight="bold", color=TEXT, loc="left", pad=6)
ax4.set_title(f"GFS {CYCLE}z  {date_label}",
              fontsize=7, color=SUBTEXT, loc="right", pad=6)

fig4.tight_layout()
fig4.savefig("moisture_flux_convergence.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 5  Convective Instability — CAPE × CIN Scatter

**CAPE** (Convective Available Potential Energy) measures fuel for convection.  
**CIN** (Convective Inhibition) measures the cap that suppresses it.  

High CAPE + low |CIN| → explosive convection.  
High CAPE + high |CIN| → suppressed but potentially severe if the cap breaks.

In [ ]:
# Crop to South America
lat_mask_sa = (lat >= -60) & (lat <= 15)
lon_mask_sa = (lon_180 >= -85) & (lon_180 <= -30)

cp_sa = CP[np.ix_(lat_mask_sa, lon_mask_sa)].ravel()
ci_sa = np.abs(CI[np.ix_(lat_mask_sa, lon_mask_sa)].ravel())
t_sa  = T2M[np.ix_(lat_mask_sa, lon_mask_sa)].ravel()

# Filter: only land points with CAPE > 0
mask = (cp_sa > 0) & np.isfinite(ci_sa) & np.isfinite(cp_sa)
cp_sa = cp_sa[mask]; ci_sa = ci_sa[mask]; t_sa = t_sa[mask]

fig5, ax5 = plt.subplots(figsize=(8, 6), facecolor=BG)
ax5.set_facecolor(BG)

sc = ax5.scatter(
    ci_sa, cp_sa,
    c=t_sa, cmap="plasma",
    vmin=15, vmax=40,
    s=2, alpha=0.3, linewidths=0,
)

# Reference thresholds
ax5.axvline(50,   color="#f7c948", linewidth=1.0, linestyle="--", alpha=0.7,
            label="CIN = 50 J/kg (moderate cap)")
ax5.axhline(1000, color="#f85149", linewidth=1.0, linestyle="--", alpha=0.7,
            label="CAPE = 1000 J/kg (moderate instability)")
ax5.axhline(2500, color="#f85149", linewidth=1.5, linestyle="-",  alpha=0.7,
            label="CAPE = 2500 J/kg (severe threshold)")

cb5 = fig5.colorbar(sc, ax=ax5, pad=0.01)
cb5.set_label("2m Temperature (°C)", fontsize=8, color=SUBTEXT)
cb5.ax.tick_params(labelsize=7, colors=SUBTEXT)
cb5.outline.set_edgecolor(BOXEDGE)

ax5.set_xlabel("|CIN| (J/kg) — inhibition strength", fontsize=9)
ax5.set_ylabel("CAPE (J/kg) — instability energy", fontsize=9)
ax5.set_xlim(0, 400); ax5.set_ylim(0, 5000)
ax5.set_title("CAPE vs |CIN| — South America grid points (colour = T2m)",
              fontsize=9, fontweight="bold", color=TEXT, loc="left")
ax5.set_title(f"GFS {CYCLE}z  {date_label}",
              fontsize=7, color=SUBTEXT, loc="right")
ax5.legend(fontsize=7, loc="upper right")

fig5.tight_layout()
fig5.savefig("cape_cin_scatter.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 6  Precipitable Water Time Series over a Region

In [ ]:
# Define a region of interest
REGIONS = {
    "Northeast Brazil": dict(lat=(-18, -1),  lon=(-47, -34)),
    "Amazon basin":     dict(lat=(-10,  5),  lon=(-75, -50)),
    "La Plata basin":   dict(lat=(-35, -20), lon=(-65, -45)),
}

n_times = len(ds_pw["pwat"].time)
forecast_h = [int(str(ds_pw["pwat"].time.values[i] - ds_pw["pwat"].time.values[0])
                  .replace("ns", "")) // (3600 * 1_000_000_000)
              for i in range(n_times)]

fig6, ax6 = plt.subplots(figsize=(13, 5), facecolor=BG)
ax6.set_facecolor(BG)

colors_reg = ["#58a6ff", "#3fb950", "#f7c948"]

for (reg_name, bounds), color in zip(REGIONS.items(), colors_reg):
    lat_lo, lat_hi = bounds["lat"]
    lon_lo, lon_hi = bounds["lon"]

    lat_mask_r = (lat >= lat_lo) & (lat <= lat_hi)
    lon_mask_r = (lon_180 >= lon_lo) & (lon_180 <= lon_hi)

    pw_series = []
    for tidx in range(n_times):
        pw_t = ds_pw["pwat"].isel(time=tidx).values[:, order]
        subset = pw_t[np.ix_(lat_mask_r, lon_mask_r)]
        pw_series.append(float(np.nanmean(subset)))

    ax6.plot(forecast_h, pw_series,
             color=color, linewidth=1.5,
             label=reg_name, marker="o", markersize=3)

ax6.set_xlabel("Forecast lead time (h)", fontsize=9)
ax6.set_ylabel("Precipitable Water (kg m⁻²)", fontsize=9)
ax6.set_title("Precipitable Water — area-mean time series",
              fontsize=10, fontweight="bold", color=TEXT, loc="left")
ax6.set_title(f"GFS {CYCLE}z  {RUN_DATE}",
              fontsize=7, color=SUBTEXT, loc="right")
ax6.legend(fontsize=8)
ax6.set_xticks(range(0, max(forecast_h) + 1, 12))

fig6.tight_layout()
fig6.savefig("pwat_timeseries.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 7  CAPE Map with Vorticity Contours — Combined diagnostic

Overlay CAPE (fill) with vorticity contours to identify where convective instability
coincides with cyclonic rotation — the most favourable environment for severe weather.

In [ ]:
levels_cape = np.array([250, 500, 1000, 1500, 2000, 2500, 3000, 4000, 5000])
cmap_cape   = copy.copy(plt.cm.YlOrRd)
cmap_cape.set_under(alpha=0); cmap_cape.set_bad(alpha=0)
norm_cape   = mcolors.BoundaryNorm(levels_cape, ncolors=cmap_cape.N, clip=False)

proj7 = ccrs.PlateCarree()
fig7, ax7 = plt.subplots(figsize=(13, 8),
                          subplot_kw={"projection": proj7}, facecolor=BG)
ax7.set_facecolor(BG)
ax7.set_extent([-85, -30, -60, 15], crs=ccrs.PlateCarree())

ax7.add_feature(cfeature.OCEAN.with_scale("50m"),  facecolor=OCEAN_C, zorder=0)
ax7.add_feature(cfeature.LAND.with_scale("50m"),   facecolor=LAND,    zorder=0)
ax7.add_feature(cfeature.COASTLINE.with_scale("50m"),
                edgecolor=COAST, linewidth=0.6, zorder=3)
ax7.add_feature(cfeature.BORDERS.with_scale("50m"),
                edgecolor=BORDER, linewidth=0.4, zorder=3)
ax7.add_feature(
    cfeature.NaturalEarthFeature("cultural",
        "admin_1_states_provinces_lines", "10m",
        facecolor="none", edgecolor=STATE, linewidth=0.3),
    zorder=3,
)

# CAPE fill
cf7 = ax7.pcolormesh(lon_180, lat, CP,
                     cmap=cmap_cape, norm=norm_cape,
                     transform=ccrs.PlateCarree(), zorder=1)

# Cyclonic vorticity contours (positive only, thinned)
ax7.contour(
    lon_180[::3], lat[::3], vort_scaled[::3, ::3],
    levels=[1, 3, 5], colors=["#58a6ff"],
    linewidths=[0.5, 0.8, 1.2], alpha=0.7,
    transform=ccrs.PlateCarree(), zorder=4,
)
# Anticyclonic vorticity contours (negative)
ax7.contour(
    lon_180[::3], lat[::3], vort_scaled[::3, ::3],
    levels=[-5, -3, -1], colors=["#f85149"],
    linewidths=[1.2, 0.8, 0.5], alpha=0.7, linestyles="--",
    transform=ccrs.PlateCarree(), zorder=4,
)

gl7 = ax7.gridlines(draw_labels=True, linewidth=0.3, color=GRID, linestyle="--")
gl7.top_labels = False; gl7.right_labels = False
gl7.xlabel_style = {"color": SUBTEXT, "fontsize": 7}
gl7.ylabel_style = {"color": SUBTEXT, "fontsize": 7}

cb7 = fig7.colorbar(cf7, ax=ax7, orientation="horizontal",
                    pad=0.05, fraction=0.03, shrink=0.7)
cb7.set_label("CAPE (J/kg)  |  blue contours = cyclonic vorticity, red = anticyclonic",
              fontsize=8, color=SUBTEXT)
cb7.ax.tick_params(labelsize=7, colors=SUBTEXT)
cb7.outline.set_edgecolor(BOXEDGE)

ax7.set_title("CAPE + Relative Vorticity — South America",
              fontsize=10, fontweight="bold", color=TEXT, loc="left", pad=6)
ax7.set_title(f"GFS {CYCLE}z  {date_label}",
              fontsize=7, color=SUBTEXT, loc="right", pad=6)

fig7.tight_layout()
fig7.savefig("cape_vorticity_combined.png", dpi=150, bbox_inches="tight")
plt.show()

---
## Exercises

1. **Upper-level diagnostics** — load `u` and `v` at 200 hPa and compute the upper-level divergence (a positive forcing for deep convection).
2. **Vorticity advection** — compute $\vec{V} \cdot \nabla\zeta$ using `numpy.gradient` to identify regions where cyclonic vorticity is being imported.
3. **Q-vectors** — derive the quasi-geostrophic Q-vector from geopotential height and temperature fields to locate regions of ageostrophic forcing for vertical motion.
4. **Animate the MFC** — loop over all 41 forecast steps and render `moisture_flux_convergence` frames, then encode to MP4 using the pattern from notebook 06.
5. **Seasonal PWAT** — load the same run date for multiple months and plot a bar chart of the area-mean PWAT trend for a region of interest.